# P8 — Warehouse Re-Slotting from Predicted Demand · Person 2 (final)
CSP (BT / FC / AC-3) · Search (Greedy / bounded UCS / A*) · layout optimisation · picker simulation on real held-out invoices.

Run: put this notebook inside the `P8_Warehouse_Person2_Final` folder tree (it needs `../src` and `../data`), `pip install -r requirements.txt`, then **Run all**.
Every section calls a step function from `src/experiments.py`; the same steps run in one go with `run_complete_project(ROOT)`.
CSP results are cached in `results/experiments/_cache` (set `FORCE = True` to recompute; ~3 min).

Honesty rules: simulated assumptions are labelled; a time limit is never reported as "no solution"; infeasibility is reported through an exact certificate; historical numbers are never presented as reproduced.

## 1. Project overview
Variables = 120 products, domains = 120 slots (10×12 grid, 4 aisles × 30 slots, low/middle/high shelves, Depot (0,0)). **Hard**: one product per slot, capacity, heavy → low shelf, cleaning↔food and cleaning↔fragile never in the same aisle, co-purchase distance ≤ `max_distance`. **Soft objective** (search/optimiser): `J = demand-weighted walking distance + 0.04 × pair excess`.

## 2. Imports and configuration

In [ ]:
import os, sys, json
import pandas as pd
try:
    from IPython.display import display
except Exception:
    display = print
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 140)

ROOT = os.getcwd()
for cand in (ROOT, os.path.dirname(ROOT), "/content/P8_Warehouse_Person2_Final", "/content/drive/MyDrive/P8_Warehouse_Person2_Final"):
    if os.path.isdir(os.path.join(cand, "src")) and os.path.isdir(os.path.join(cand, "data")):
        ROOT = cand; break
else:
    raise FileNotFoundError("Run inside the P8_Warehouse_Person2_Final folder (needs src/ and data/).")
os.chdir(ROOT); sys.path.insert(0, ROOT)

from src import experiments as EXP
from src.warehouse import CATEGORIES, DEPOT, manhattan
from src.constraints import categories_incompatible, build_pair_index, aisle_capacity_certificate
from src.validation import validate_layout, report_summary, hard_basic_ok
from src.picker_simulator import route_distance, evaluate_layout
FORCE = False
print("Project root:", ROOT)

## 3. Load and validate data
Schema detection handles `Invoice` / `InvoiceNo`; ids are normalised (`123.0` → `123`); invalid / duplicate rows are dropped and counted; co-purchase strength keeps full float precision. Files on disk are never modified.

In [ ]:
ctx = EXP.prepare(ROOT)
print(json.dumps(ctx["data_report"], indent=2))
display(ctx["data"]["forecasts"].head(3)); display(ctx["data"]["pairs"].head(3)); display(ctx["data"]["orders"].head(3))

## 4. Warehouse and slot generation
Generated (the dataset has no geometry): rows 0-3 low shelf, 4-6 middle, 7-9 high; `aisle = col // 3`; capacity 100; Manhattan distance.

In [ ]:
slots, SBY = ctx["slots"], ctx["sby"]
print("Slots:", len(slots), "| unique ids:", len({s.slot_id for s in slots}))
display(pd.DataFrame([{"level": s.level, "aisle": s.aisle + 1} for s in slots]).value_counts().unstack(fill_value=0))
assert len(slots) == 120 and manhattan(DEPOT, (2, 3)) == 5

## 5. Product and category constraints
Category / heavy flag / required capacity are **simulated** (seed 42). Rules: cleaning↔food and cleaning↔fragile are incompatible (not in the same aisle).

In [ ]:
products, PBY = ctx["products"], ctx["pby"]
print(pd.Series([p["category"] for p in products]).value_counts().to_dict(), "| heavy:", sum(p["heavy"] for p in products), "| low-shelf slots:", sum(s.level == "low" for s in slots))
display(pd.DataFrame([{"A": a, "B": b, "incompatible": categories_incompatible(a, b)} for i, a in enumerate(CATEGORIES) for b in CATEGORIES[i:]]))
print("Exact aisle-rule certificate, all 120 products:", ctx["cert"])

**Feasibility finding.** The certificate is an exact test of the category-aisle rule alone: cleaning products need whole aisles free of food/fragile. With the real counts above, no split of the 4 aisles holds both groups, so the strict 120-product problem is **infeasible**. The strict rule is kept; a relaxed-rule diagnostic is run separately in §9.

## 6. Co-purchase constraint preparation

In [ ]:
pairs = ctx["pairs"]
print("Pairs:", len(pairs), "| max_distance:", pd.Series([p[2] for p in pairs]).value_counts().to_dict(), "| strength dtype:", type(pairs[0][3]).__name__)
display(pd.DataFrame(pairs[:8], columns=["product_a", "product_b", "max_distance", "strength"]))
deg = pd.Series({k: len(v) for k, v in build_pair_index(pairs).items()}).sort_values(ascending=False); print("Hubs:", deg.head(5).to_dict())

## 7. Layout validation
`validate_layout` is the single validator used everywhere. It checks completeness, unknown products, slot ids, duplicates, capacity, heavy shelf, incompatible aisles, pair distances and total excess.

In [ ]:
from src.optimization import build_demand_layout
nn_demo = build_demand_layout(products, slots, "demand")
print(json.dumps(report_summary(validate_layout(products, nn_demo, slots, pairs)), indent=2))
bad = dict(nn_demo); a, b = list(bad)[:2]; bad[a] = bad[b]
assert validate_layout(products, bad, slots, pairs)["duplicate_slots"]
assert validate_layout(products, {k: v for k, v in list(nn_demo.items())[:-1]}, slots, pairs)["missing_products"]
print("Validator spot checks passed (the full test-suite runs in §16).")

## 8. CSP algorithms
Shared pre-indexed model (`csp_model.py`): domains are bit masks (bit 0 = slot nearest the Depot), pair index, distance balls and aisle masks are precomputed, domain changes are undone from a trail.
* **BT** – static demand order, Depot-nearest values first, checks against assigned variables only.
* **BT+FC** – MRV (degree tie-break), incremental forward checking, plus a pigeonhole check for heavy products.
* **BT+FC+AC-3** – FC plus genuine AC-3 over pair-distance arcs and incompatible-category arcs (queue, revise, neighbour re-queue, wipe-out detection, counters). All-different is left to FC (documented).

Statuses: SOLVED · NO SOLUTION · NODE LIMIT REACHED · TIME LIMIT REACHED · INFEASIBLE BY CERTIFICATE. Smoke test first (30 products, 10 strongest pairs):

In [ ]:
smoke = EXP.run_csp_smoke(ctx, force=FORCE)
display(smoke[["Algorithm", "Pair constraints", "Status", "Nodes", "Backtracks", "Constraint checks", "Time (s)", "Valid layout"]])
assert (smoke[smoke.Algorithm != "BT"].Status == "SOLVED").all(), "FC / AC-3 smoke test must pass before the large experiment"

## 9. CSP experiments (60 / 90 / 120 products)
Identical problem per row group (same top-N products, strongest pairs with both endpoints inside the subset, limits 20 s / 2 M nodes). Protocol A: 20/30/40 pairs; protocol B: 30 pairs. For 120 products the strict problem is proven infeasible, so no search is run there. A separate **diagnostic** relaxes only the aisle rule.

In [ ]:
csp_df, diag_df = EXP.run_csp(ctx, force=FORCE)
cols = ["Products", "Algorithm", "Protocol", "Pair constraints", "Status", "Nodes", "Backtracks", "Constraint checks", "Domain reductions", "AC-3 revisions", "Time (s)", "Valid layout"]
display(csp_df[cols])
print("DIAGNOSTIC (aisle rule relaxed, 120 products, 40 pairs):"); display(diag_df[cols + ["pair_violations", "aisle_violations"]])

## 10. Search algorithms
**State** complete layout · **Action** swap two products' slots (one top-30-demand product with a partner in the 40 slots nearest the Depot, or another top-30 product) · **Cost** 1/swap · depth ≤ 6 · 300 expansions · duplicates pruned by layout signature. **J = WWD + 0.04 × pair excess** (predicted demand, training pairs; no test invoices). **Greedy**: priority J. **Bounded UCS**: priority g (J breaks ties). **A\***: f = g + h, h = (J − target)/best single-swap gain at the root — not admissible, so no optimality claim. Goal: J ≤ 90 % of the initial J. Each child keeps unique slots, capacity, heavy→low and never adds incompatible-aisle pairs.

In [ ]:
obj = ctx["objective"]; st0 = obj.state_from_layout(build_demand_layout(products, slots, "demand"))
print(f"Start (NN demand): J={obj.total(st0):.4f}  WWD={obj.wwd(st0):.4f}  pair excess/violations={obj.excess(st0)}  incompatible-aisle pairs={obj.aisle_conflicts(st0)}")

## 11. Search experiments

In [ ]:
search_df = EXP.run_search(ctx)
display(search_df.drop(columns=["heuristic_delta"]).T)

## 12. Layout optimisation
One optimiser: swap-based simulated annealing on the same J over all 120 products, starting from the NN-demand layout. Hard slot rules are checked on every move (infeasible swaps rejected) and incompatible-aisle pairs may never increase. Pair distances stay soft here and remaining violations are reported separately. No test invoices are used.

In [ ]:
layouts, sa_info = EXP.build_layouts(ctx)
print(sa_info)
assert all(hard_basic_ok(validate_layout(products, l, slots, pairs)) for l in layouts.values())

## 13. Picker-distance simulation
Route (identical for all layouts): Depot → nearest unvisited product (Manhattan, deterministic tie-break) → … → Depot; one visit per distinct product per invoice. A nearest-neighbour heuristic, **not** an optimal TSP tour. 1,000 development invoices are excluded from the reserved holdout (same split as before).

In [ ]:
assert route_distance([(2, 3, "x")]) == 10 and route_distance([(0, 1, "a"), (0, 3, "b")]) == 6
print("Simulator checks passed | development invoices:", len(ctx["dev_inv"]), "| reserved holdout invoices:", len(ctx["hold_inv"]),
      "| holdout lines:", sum(len(v) for v in ctx["hold_inv"].values()))

## 14. Four-layout comparison (reserved holdout)
Random (seed 42) · Last-week demand · NN demand · NN + Swaps. Same products, warehouse, holdout invoices and route model for all.

In [ ]:
four_df, dev_df = EXP.evaluate_layouts(ctx)
display(four_df.sort_values("Average distance per invoice").reset_index(drop=True))
EXP.export_layouts(ctx)
print("Historical reference (earlier notebook, NOT recomputed): initial 29.344 / 121 violations -> joint optimisation 27.006 / 22 violations.")

**Reading the table.** *Basic rules OK* (unique slots, capacity, heavy→low) holds for all four layouts. *Strictly valid* is False for all four: hard pair constraints remain violated, and because the aisle rule is infeasible for 120 products incompatible-aisle pairs remain too. Layouts are never labelled fully valid when they are not.

## 15. Visualisations
Dark theme, 300-DPI PNG, one colour per category / algorithm / layout, data for every figure in `figures/data/`.

In [ ]:
EXP.make_figures(ctx)
missing = [f for f in EXP.EXPECTED_FIGURES if not os.path.exists(f"figures/{f}")]
print(missing or f"all {len(EXP.EXPECTED_FIGURES)} figures written")

## 16. Final validation (automated QA + unit tests)
Every test actually executes. A FAIL is reported as FAIL; the two strict-scope FAILs are the documented consequence of the infeasible aisle rule and soft pair constraints.

In [ ]:
qa_df = EXP.run_qa_step(ctx)
display(qa_df)
print(qa_df.Result.value_counts().to_dict())

## 17. Export results and reports

In [ ]:
EXP.write_all_reports(ctx)
for root, dirs, files in os.walk(ROOT):
    dirs[:] = [d for d in dirs if d not in ("_cache", "__pycache__")]
    for f in sorted(files):
        print(os.path.relpath(os.path.join(root, f), ROOT), os.path.getsize(os.path.join(root, f)), "bytes")
# One-call alternative (re-runs everything from the cache):  EXP.run_complete_project(ROOT)